In [ ]:
import os
import xarray as xr
import numpy as np
from cdo import Cdo
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import tempfile
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from eocene.plots.helper_functions import load_grib_var_spec, compare_maps, compare_maps_cl, plot_spectral_sanity

from eocene.oifs.eoceneOIFS import EoceneOIFS

cdo = Cdo()

In [ ]:
eocene = EoceneOIFS(
    idir="/path/to/ECE4-DATA",
    odir="/path/to/output",
    herold="/path/to/herold-et-al-2014",
    startdate="19900101",
)

In [ ]:
landsea = xr.open_dataset(eocene.prepare_herold(flag="landsea_mask"))
orog = xr.open_dataset(eocene.prepare_herold(flag="orography"))
sd_orog = xr.open_dataset(eocene.prepare_herold(flag="sd_orography"))
bathy = xr.open_dataset(eocene.prepare_herold(flag="bathymetry"))

In [ ]:
orog_eocene = orog["orography"]
plt.figure(figsize=(6, 4))
orog_eocene.plot()
plt.title("Herold orography")
plt.show()

In [ ]:
bathymetry_eocene = bathy["bathymetry"]
plt.figure(figsize=(6, 4))
bathymetry_eocene.plot()
plt.title("Herold bathymetry")
plt.show()

In [ ]:
landsea_eocene = landsea["landsea_mask"]
plt.figure(figsize=(6, 4))
landsea_eocene.plot()
plt.title("Herold landsea_mask")
plt.show()

In [ ]:
lsm_present = eocene.prepare_landsea_mask_present()
plt.figure(figsize=(6, 4))
lsm_present.plot()
plt.title("IFS Present-Day LSM")
plt.show()

## ICMGGECE4INIT

In [ ]:
orig = os.path.join(eocene.idir_init, "ICMGGECE4INIT")
new = os.path.join(eocene.odir_init, "ICMGGECE4INIT")

In [ ]:
# Sub-grid orography
compare_maps("sdor", orig, new)  # subgrid orography
compare_maps("sdfor", orig, new)  # subgrid f? orography
compare_maps("slor", orig, new)  # slope
compare_maps("isor", orig, new)  # anysotropy
compare_maps("anor", orig, new)  # angle

In [ ]:
# Land-sea mask
compare_maps("lsm", orig, new)  # land-sea mask
compare_maps("lsmgrd", orig, new)

In [ ]:
# Charnok
compare_maps("chnk", orig, new)

In [ ]:
# Albedo variables
compare_maps("al", orig, new)
compare_maps("aluvp", orig, new)
compare_maps("aluvd", orig, new)
compare_maps("alnip", orig, new)
compare_maps("alnid", orig, new)
compare_maps("aluvpi", orig, new)
compare_maps("aluvpv", orig, new)
compare_maps("aluvpg", orig, new)
compare_maps("alnipi", orig, new)
compare_maps("alnipv", orig, new)
compare_maps("alnipg", orig, new)

In [ ]:
# Other variables
compare_maps("cl", orig, new)
compare_maps("dl", orig, new)
compare_maps("slt", orig, new)  # soil texture
compare_maps("licd", orig, new)
compare_maps("sd", orig, new)  # snow depth

## Vegetation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.colors import ListedColormap, BoundaryNorm


def plot_veg_classes_cartopy(gribfile):

    tvl = load_grib_var_spec(gribfile, "tvl").squeeze()
    tvh = load_grib_var_spec(gribfile, "tvh").squeeze()
    lsm = load_grib_var_spec(gribfile, "lsm").squeeze()

    # mask ocean
    tvl = tvl.where(lsm > 0)
    tvh = tvh.where(lsm > 0)

    # mask "no vegetation" (0 values)
    tvl = tvl.where(tvl != 0)
    tvh = tvh.where(tvh != 0)

    # wrap longitude to -180 -> 180
    tvl = tvl.assign_coords(lon=((tvl.lon + 180) % 360) - 180).sortby("lon")
    tvh = tvh.assign_coords(lon=((tvh.lon + 180) % 360) - 180).sortby("lon")

    classes = [1, 2, 3, 4, 5, 6, 7, 8]
    labels = {
        1: "Evergreen broadleaf trees",
        2: "Evergreen needleleaf trees",
        3: "Deciduous broadleaf",
        4: "Deciduous needleleaf",
        5: "Tall grass",
        6: "Short grass",
        7: "Semidesert",
        8: "Tundra",
    }

    colors = ["#1b7837", "#5aae61", "#a6dba0", "#d9f0d3", "#fee08b", "#fdae61", "#f46d43", "#d73027"]

    cmap = ListedColormap(colors)
    cmap.set_bad("white")  # masked values appear white

    bounds = np.arange(0.5, 9.5, 1)
    norm = BoundaryNorm(bounds, cmap.N)

    proj = ccrs.PlateCarree()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), subplot_kw={"projection": proj})

    # plot high vegetation
    im1 = tvh.plot(ax=axes[0], cmap=cmap, norm=norm, add_colorbar=False, transform=proj)
    # axes[0].coastlines()
    axes[0].set_global()
    axes[0].set_title("a. High vegetation (tvh)")

    # plot low vegetation
    im2 = tvl.plot(ax=axes[1], cmap=cmap, norm=norm, add_colorbar=False, transform=proj)
    # axes[1].coastlines()
    axes[1].set_global()
    axes[1].set_title("b. Low vegetation (tvl)")

    # add gridlines with labels
    for ax in axes:
        gl = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.5)
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {"size": 10}
        gl.ylabel_style = {"size": 10}
        gl.xlocator = plt.FixedLocator(np.arange(-180, 181, 60))
        gl.ylocator = plt.FixedLocator(np.arange(-90, 91, 30))

    # horizontal colorbar
    fig.subplots_adjust(bottom=0.25)
    cax = fig.add_axes([0.2, 0.08, 0.6, 0.04])
    cbar = fig.colorbar(im2, cax=cax, orientation="horizontal")
    cbar.set_ticks(classes)
    cbar.set_ticklabels([labels[i] for i in classes])
    cbar.set_label("Vegetation type")
    cbar.ax.set_xticklabels([labels[i] for i in classes], rotation=35, ha="right")

    plt.show()


new = os.path.join(eocene.odir_init, "ICMGGECE4INIT")
plot_veg_classes_cartopy(new)

## ICMCLECE4 (climate fields)

In [ ]:
orig = os.path.join(eocene.idir_climate, "ICMCLECE4")
new = os.path.join(eocene.odir_climate, "ICMCLECE4")

compare_maps_cl("al", orig, new)
compare_maps_cl("aluvp", orig, new)
compare_maps_cl("aluvd", orig, new)
compare_maps_cl("alnip", orig, new)
compare_maps_cl("alnid", orig, new)
compare_maps_cl("lai_lv", orig, new)
compare_maps_cl("lai_hv", orig, new)

In [ ]:
orig = os.path.join(eocene.idir_climate, "bare_soil_albedos.grb")
new = os.path.join(eocene.odir_climate, "bare_soil_albedos.grb")

compare_maps_cl("var117", orig, new)
compare_maps_cl("var118", orig, new)
compare_maps_cl("var119", orig, new)
compare_maps_cl("var120", orig, new)

## Aerosols

In [ ]:
aerosols_ds = xr.open_dataset("/path/to/output-with-aerosols/oifs/ifsdata/aerosol_cams_climatology_43R3a.nc")

In [ ]:
import matplotlib as mpl

mpl.rcParams.update(
    {
        "font.size": 20,  # base font
        "axes.titlesize": 28,  # title
        "axes.labelsize": 24,  # axis labels
        "xtick.labelsize": 20,
        "ytick.labelsize": 20,
        "legend.fontsize": 20,
    }
)

In [ ]:
# --- Sum all mineral dust bins ---
dust_total = aerosols_ds["Mineral_Dust_bin1"] + aerosols_ds["Mineral_Dust_bin2"] + aerosols_ds["Mineral_Dust_bin3"]

# --- Time mean if needed ---
if "month" in dust_total.dims:
    dust_mean = dust_total.mean(dim="month")
else:
    dust_mean = dust_total

# --- Add cyclic point for global map ---
dust_cyc, lon_cyc = add_cyclic_point(dust_mean, coord=dust_mean.lon)

In [ ]:
# --- Create figure ---
fig, ax = plt.subplots(figsize=(18, 10), dpi=100, subplot_kw={"projection": ccrs.PlateCarree()})

# --- Plot ---
cf = ax.contourf(lon_cyc, dust_mean.lat, dust_cyc, transform=ccrs.PlateCarree(), cmap="RdBu_r", extend="both")

# --- Map features ---
ax.add_feature(cfeature.LAND, facecolor="lightgray", zorder=0)
ax.add_feature(cfeature.COASTLINE, color="black", linewidth=1.5)
ax.add_feature(cfeature.BORDERS, color="black", linewidth=1.0)

# --- Lat/Lon ticks ---
ax.set_xticks(np.arange(-180, 181, 60), crs=ccrs.PlateCarree())
ax.set_yticks(np.arange(-90, 91, 30), crs=ccrs.PlateCarree())

ax.set_xlabel("Longitude", fontsize=20)
ax.set_ylabel("Latitude", fontsize=20)
ax.tick_params(labelsize=18)

# --- Title ---
# ax.set_title("Total Mineral Dust", fontsize=26, fontweight="bold")

# --- Create separate colorbar axis ---
cax = fig.add_axes([0.125, 0.04, 0.775, 0.025])
# [left, bottom, width, height] in figure coordinates

cbar = fig.colorbar(cf, cax=cax, orientation="horizontal")


units = dust_mean.attrs.get("units", "")
cbar.set_label(f"Mineral Dust ({units})", fontsize=20)
cbar.ax.tick_params(labelsize=18)

plt.show()

In [ ]:
orig = os.path.join(eocene.idir_init, "ICMSHECE4INIT")
new = os.path.join(eocene.odir_init, "ICMSHECE4INIT")

plot_spectral_sanity("z", orig, new)  # geopotential